# Model Building

**Models**: Logistic Regression, Decision Tree, Random Forest, SVC and KNN

### 1 - Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (precision_recall_curve,accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix)
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

### 2 - Load Cleaned Data

In [ ]:
df = pd.read_csv('data.csv')
print(f" Shape: {df.shape}")
df.head()

### 3 - Prepare Features and Target

Prepare the features to be used in the train/tests

In [ ]:
all_features = ['age', 'sex', 'bmi', 'systolic_pressure', 'diastolic_pressure',
                'glucose', 'hba1c', 'cholesterol', 'hdl_cholesterol',
                'ldl_cholesterol', 'triglycerides', 'smoking',
                'physical_activity', 'family_history_diabetes', 'family_history_hypertension']

#x for features and y for targets, one for diabetes and one for hypertension
X = df[all_features].copy()
y_diabetes = df['diabetes_risk']
y_hypertension = df['hypertension_risk']

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Targets:")
print(f"- Diabetes: {y_diabetes.sum()} positives ({y_diabetes.mean() * 100:.1f}%)")
print(f"- Hypertension: {y_hypertension.sum()} positives ({y_hypertension.mean() * 100:.1f}%)")

#split 80 for train and 20 for test to allign the train
X_train, X_test, y_diab_train, y_diab_test, y_hyp_train, y_hyp_test = train_test_split(
    X, y_diabetes, y_hypertension, test_size=0.2, random_state=67, stratify=y_diabetes
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

#only contiguos features, binary dont need scale
numeric_features = ['age', 'bmi', 'systolic_pressure', 'diastolic_pressure',
                    'glucose', 'hba1c', 'cholesterol', 'hdl_cholesterol',
                    'ldl_cholesterol', 'triglycerides']

#rescale the numeric features so all values are on the same scale 
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

# SMOTE to SVM and KNN
smote_diab = SMOTE(random_state=67)
X_train_balanced_diab, y_diab_train_balanced = smote_diab.fit_resample(X_train_scaled, y_diab_train)

smote_hyp = SMOTE(random_state=67)
X_train_balanced_hyp, y_hyp_train_balanced = smote_hyp.fit_resample(X_train_scaled, y_hyp_train)

### 4 - Train Models

Train models, train models for diabetes and models for hypertension

In [ ]:
base_models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=67, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=67, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=67),
    'SVM': SVC(class_weight='balanced', probability=True, random_state=67),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

models = base_models
diabetes_models = {}
diabetes_thresholds = {}
hypertension_models = {}
hypertension_thresholds = {}
diabetes_results = []
hypertension_results = []

#### 4.1 - Diabetes models


In [ ]:
#diabetes models
for name, model_template in base_models.items():
    model = clone(model_template)
    
    #SVM and KNN use the SMOTE
    if name in ['SVM', 'KNN']:
        model.fit(X_train_balanced_diab, y_diab_train_balanced)
    else:
        model.fit(X_train_scaled, y_diab_train)
    
    #get prob pos
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    #threshold tuning
    precisions, recalls, thresholds = precision_recall_curve(y_diab_test, y_proba)

    #f1 for each threshold 
    f1_scores = []
    for prec, recal in zip(precisions, recalls):
        if prec + recal == 0: #avoid division by 0
            f1_scores.append(0)
        else:
            f1_scores.append(2 * prec * recal / (prec + recal))

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    
    #predictions
    y_pred = (y_proba >= best_threshold).astype(int)
    
    #final results
    acc = accuracy_score(y_diab_test, y_pred)
    prec = precision_score(y_diab_test, y_pred)
    rec = recall_score(y_diab_test, y_pred)
    f1 = f1_score(y_diab_test, y_pred)
    auc = roc_auc_score(y_diab_test, y_proba) 
    
    #store results
    diabetes_models[name] = model
    diabetes_thresholds[name] = best_threshold 
    
    diabetes_results.append({
        'Model': name, 
        'Threshold': best_threshold, 
        'Accuracy': acc, 
        'Precision': prec,
        'Recall': rec, 
        'F1-Score': f1, 
        'ROC-AUC': auc
    })
    
print(f"Diabetes models train done")

#### 4.2 - Hypertension models


In [ ]:
# Hypertension models
for name, model_template in base_models.items():
    model = clone(model_template)
    
    if name in ['SVM', 'KNN']:
        model.fit(X_train_balanced_hyp, y_hyp_train_balanced)
    else:
        model.fit(X_train_scaled, y_hyp_train)
    

    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    #threshold 
    precisions, recalls, thresholds = precision_recall_curve(y_hyp_test, y_proba)
    
    f1_scores = []
    for prec, recal in zip(precisions, recalls):
        if prec + recal == 0:
            f1_scores.append(0)
        else:
            f1_scores.append(2 * prec * recal / (prec + recal))
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    
    y_pred = (y_proba >= best_threshold).astype(int)
    
    #final res
    acc = accuracy_score(y_hyp_test, y_pred)
    prec = precision_score(y_hyp_test, y_pred)
    rec = recall_score(y_hyp_test, y_pred)
    f1 = f1_score(y_hyp_test, y_pred)
    auc = roc_auc_score(y_hyp_test, y_proba)
    
    hypertension_models[name] = model
    hypertension_thresholds[name] = best_threshold 
    
    hypertension_results.append({
        'Model': name, 
        'Threshold': best_threshold, 
        'Accuracy': acc, 
        'Precision': prec,
        'Recall': rec, 
        'F1-Score': f1, 
        'ROC-AUC': auc
    })

print(f"Hypertension models train done")

### 5 - Model Comparison

Analysis so we can know the best model by ROC-AUC metric

In [ ]:
diab_df = pd.DataFrame(diabetes_results)
hyp_df = pd.DataFrame(hypertension_results)

#select the best model using ROC-AUC as the main metric
best_diab_model = diab_df.loc[diab_df['ROC-AUC'].idxmax(), 'Model']
best_diab_auc = diab_df['ROC-AUC'].max()
best_diab_f1 = diab_df.loc[diab_df['ROC-AUC'].idxmax(), 'F1-Score']

best_hyp_model = hyp_df.loc[hyp_df['ROC-AUC'].idxmax(), 'Model']
best_hyp_auc = hyp_df['ROC-AUC'].max()
best_hyp_f1 = hyp_df.loc[hyp_df['ROC-AUC'].idxmax(), 'F1-Score']

print("Diabetes Models Performance:\n")
print(diab_df[['Model', 'ROC-AUC', 'F1-Score', 'Recall', 'Precision']].to_string(index=False))

print("\n")
print("Hypertension Models Performance:\n")
print(hyp_df[['Model', 'ROC-AUC', 'F1-Score', 'Recall', 'Precision']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [(axes[0], diab_df, 'Diabetes Risk - ROC-AUC'), (axes[1], hyp_df,  'Hypertension Risk - ROC-AUC')]:
    df_sorted = df.sort_values('ROC-AUC', ascending=True)
    
    colors = ["#F46D43", "#FDC675", "#FEFEBD", "#BFE37A", "#66BD63"]
    bars = ax.barh(df_sorted['Model'], df_sorted['ROC-AUC'], color=colors)
    
    ax.set_xlabel('ROC-AUC Score')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, 1.0) #end ant 1
    
    for bar, val in zip(bars, df_sorted['ROC-AUC']):
        ax.text(val, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')


plt.tight_layout()
plt.show()

print("Best Model")
print("Diabetes:")
print(f"Best Model: {best_diab_model}")
print(f"ROC-AUC: {best_diab_auc:.4f}")
print(f"F1-Score: {best_diab_f1:.4f}")
print("\nHypertension")
print(f"Best Model: {best_hyp_model}")
print(f"ROC-AUC: {best_hyp_auc:.4f}")
print(f"F1-Score: {best_hyp_f1:.4f}")

### 6 - Results

Different graphs for the results


#### 6.1 - Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#diabetes
best_diab_thresh = diabetes_thresholds[best_diab_model]
y_diab_proba = diabetes_models[best_diab_model].predict_proba(X_test_scaled)[:, 1]

#use the threshold to get the prob
y_diab_pred = (y_diab_proba >= best_diab_thresh).astype(int)

#calc and draw matrix
conf_matrix_diab = confusion_matrix(y_diab_test, y_diab_pred)
sns.heatmap(conf_matrix_diab, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'])

axes[0].set_title(f'Confusion Matrix for Diabetes\n({best_diab_model})', fontweight='bold')
axes[0].set_xlabel('Prediction Value')
axes[0].set_ylabel('Real Value')


#same but for hyp
best_hyp_thresh = hypertension_thresholds[best_hyp_model]
y_hyp_proba = hypertension_models[best_hyp_model].predict_proba(X_test_scaled)[:, 1]

# Aplicar o threshold
y_hyp_pred = (y_hyp_proba >= best_hyp_thresh).astype(int)

# Calcular e desenhar a matriz
conf_matrix__hyp = confusion_matrix(y_hyp_test, y_hyp_pred)
sns.heatmap(conf_matrix__hyp, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'])

axes[1].set_title(f'Confusion Matrix for Hypertension\n({best_hyp_model})', fontweight='bold')
axes[1].set_xlabel('Prediction Value')
axes[1].set_ylabel('Real Value')

plt.tight_layout()
plt.show()

#### 6.2 - ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, models_dict, y_test, title in [
    (axes[0], diabetes_models, y_diab_test, 'Diabetes'),
    (axes[1], hypertension_models, y_hyp_test, 'Hypertension')
]:
    for name, model in models_dict.items():
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        false_pos_rate, true_pos_rate, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        
        ax.plot(false_pos_rate, true_pos_rate, label=f'{name} (AUC={auc:.3f})')

    #diagonal
    ax.plot([0, 1], [0, 1], '--', linewidth=0.8)
    
    #labels
    ax.set_xlabel('False positives rate')
    ax.set_ylabel('True positives rate')
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

#### 6.3 - Feature Importance

Check each feature importance for the best model

In [ ]:
#Diabetes
if best_diab_model == 'Random Forest':
    model = diabetes_models['Random Forest']
    importances = model.feature_importances_
    
elif best_diab_model == 'Logistic Regression':
    model = diabetes_models['Logistic Regression']
    importances = np.abs(model.coef_[0])
    
elif best_diab_model == 'Decision Tree':
    model = diabetes_models['Decision Tree']
    importances = model.feature_importances_
    
else:  # SVM or KNN
    model = diabetes_models[best_diab_model]
    print(f"Calculating permutation importance for {best_diab_model} (Diabetes)...")
    perm_importance = permutation_importance(model, X_test_scaled, y_diab_test,
        n_repeats=10, random_state=67, scoring='roc_auc')
    importances = perm_importance.importances_mean

#normalize
importances = importances / importances.max()
feature_names = all_features
indices = np.argsort(importances)[::-1]

#graph fot diabetes
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances[indices[:10]])
plt.yticks(range(10), [feature_names[i] for i in indices[:10]])
plt.xlabel('Feature Importance')
plt.title(f'Diabetes Risk ({best_diab_model})', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()



# Hypertension
if best_hyp_model == 'Random Forest':
    model_hyp = hypertension_models['Random Forest']
    importances_hyp = model_hyp.feature_importances_
    
elif best_hyp_model == 'Logistic Regression':
    model_hyp = hypertension_models['Logistic Regression']
    importances_hyp = np.abs(model_hyp.coef_[0])
    
elif best_hyp_model == 'Decision Tree':
    model_hyp = hypertension_models['Decision Tree']
    importances_hyp = model_hyp.feature_importances_
    
else:  # SVM or KNN 
    model_hyp = hypertension_models[best_hyp_model]
    print(f"Calculating permutation importance for {best_hyp_model} (Hypertension)...")
    perm_importance = permutation_importance(model_hyp, X_test_scaled, y_hyp_test,n_repeats=10, random_state=67, scoring='roc_auc')
    importances_hyp = perm_importance.importances_mean  

#same, normalize
importances_hyp = importances_hyp / importances_hyp.max()
indices_hyp = np.argsort(importances_hyp)[::-1]

#graph fot hypertension
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances_hyp[indices_hyp[:10]])
plt.yticks(range(10), [feature_names[i] for i in indices_hyp[:10]])
plt.xlabel('Feature Importance')
plt.title(f'Hypertension Risk ({best_hyp_model})', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### 7 - Export the data 

Export the data to a .pkl file

In [ ]:
#export the selected best models and scaler for the webapp
os.makedirs("app/model", exist_ok=True)

bundle = {
    "scaler": scaler,
    "diabetes": diabetes_models[best_diab_model],
    "hypertension": hypertension_models[best_hyp_model]
}

joblib.dump(bundle, "app/model/models.pkl")
print("Model saved")
